# Character-Level Text Generation
That is:

It receives JSON packet lines as raw text.

It treats each character as a token.

The RNN learns all character sequences sequentially.

New packets are created as text in the same format.

## STEP 1 — Load vocabulary mappings

In [1]:
import torch
import torch.nn as nn
import json 
# ---- LOAD VOCABULARY ----
with open("exp1_rnn_vocab.json", "r") as f:
    vocab = json.load(f)

chars = vocab["chars"]
stoi = vocab["stoi"]
itos = {int(k): v for k, v in vocab["itos"].items()} # keys must be int



## STEP 2 — Recreate the model architecture

In [2]:
class CharRNN(nn.Module):
    def __init__(self, vocab, hidden=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, 128)
        self.lstm = nn.LSTM(128, hidden, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x, h=None):
        x = self.embed(x)
        out, h = self.lstm(x, h)
        logits = self.fc(out)
        return logits, h


## STEP 3 — Load the model weights

In [3]:
model = CharRNN(len(chars))
model.load_state_dict(torch.load("exp1_rnn_model.pth", map_location="cpu"))
model.eval()

print("Model and vocabulary loaded successfully!")

Model and vocabulary loaded successfully!


## STEP 4 — Define encode/decode again

In [4]:
# ---- ENCODE / DECODE ----

def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

def decode(idx_list):
    return "".join([itos[i] for i in idx_list])

## STEP 5 — Generate traffic again

In [5]:
def generate(model, start="\n", length=5000, temperature=0.7):
    model.eval()
    chars_out = [stoi[c] for c in start]
    h = None
    x = torch.tensor([chars_out[-1]]).unsqueeze(0)
    for _ in range(length):
        logits, h = model(x, h)
        logits = logits[:, -1, :] / temperature
        prob = torch.softmax(logits, dim=-1).squeeze()
        ix = torch.multinomial(prob, 1).item()
        chars_out.append(ix)
        x = torch.tensor([[ix]])
    return decode(chars_out)

In [6]:
# length was 18739 for 10 minutes
# length is approx 3 times for 30 minutes
num_char = 18739 * 3
generated_text = generate(model, start="\n", length=num_char, temperature=0.7)

raw_file_path = f"..\Generated_Traffic\TXT_files\RNN_Exp1_Trial_raw_generated_30_minutes.txt"
with open(raw_file_path, "w", encoding="utf-8") as f:
    f.write(generated_text)
    
print(generated_text)


<>:6: SyntaxWarning: invalid escape sequence '\G'
<>:6: SyntaxWarning: invalid escape sequence '\G'
C:\Users\nkelesoglu\AppData\Local\Temp\ipykernel_25052\286293293.py:6: SyntaxWarning: invalid escape sequence '\G'
  raw_file_path = f"..\Generated_Traffic\TXT_files\RNN_Exp1_Trial_raw_generated_30_minutes.txt"



{"time": "208.217291", "src": "0x1de6", "dst": "0xfffc", "protocol": "ZigBee", "length": "21", "info": "ZCL: Read Attributes Response, Seq: 26"}
{"time": "33.038379", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 238"}
{"time": "414.64038", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 203"}
{"time": "50.73404", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "61", "info": "ZCL: Read Attributes Response, Seq: 42"}
{"time": "33.34732", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 29"}
{"time": "39.72517", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read Attributes Response, Seq: 112"}
{"time": "198.317923", "src": "0x1de6", "dst": "0xd7a7", "protocol": "ZigBee HA", "length": "69", "info": "ZCL: Read At

## STEP 6 — Open raw output → clean → save as JSON

In [7]:
def clean_rnn_output(raw_text): 
    packets = [] 
    lines = raw_text.split("\n") 

    for line in lines: 
        try: 
            line_fixed = line.strip().rstrip(",") 
            pkt = json.loads(line_fixed) 
            packets.append(pkt) 
        except: 
            continue # half packets are automatically skipped 



    return packets, packets_30min


N = list(range(1,2)) # number of trial
packets_30min  = []
for n in N:
    raw_file_path = f"..\Generated_Traffic\TXT_files\RNN_Exp1_Trial_raw_generated_30_minutes.txt"

    # read Raw text file
    with open(raw_file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    packets_all, packets_30min = clean_rnn_output(raw_text)

    clean_all_path = f"../Generated_Traffic\JSON_files\RNN_Exp1_Trial_{n}_generated_30_minutes.json"
    with open(clean_all_path, "w", encoding="utf-8") as f:
        json.dump(packets_all, f, indent=2)



<>:22: SyntaxWarning: invalid escape sequence '\G'
<>:35: SyntaxWarning: invalid escape sequence '\J'
<>:22: SyntaxWarning: invalid escape sequence '\G'
<>:35: SyntaxWarning: invalid escape sequence '\J'
C:\Users\nkelesoglu\AppData\Local\Temp\ipykernel_25052\1754936467.py:22: SyntaxWarning: invalid escape sequence '\G'
  raw_file_path = f"..\Generated_Traffic\TXT_files\RNN_Exp1_Trial_raw_generated_30_minutes.txt"
C:\Users\nkelesoglu\AppData\Local\Temp\ipykernel_25052\1754936467.py:35: SyntaxWarning: invalid escape sequence '\J'
  clean_all_path = f"../Generated_Traffic\JSON_files\RNN_Exp1_Trial_{n}_generated_30_minutes.json"
